In [ ]:


class FlexibleMLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dims=[64, 32, 16], output_dim=1):
        """
        Flexible MLP that can vary in both depth and width
        hidden_dims: list of integers specifying the width of each hidden layer
        """
        super(FlexibleMLP, self).__init__()

        # Create layers dynamically based on hidden_dims
        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.Sigmoid())
            prev_dim = hidden_dim

        # Output layer
        layers.append(nn.Linear(prev_dim, output_dim))
        layers.append(nn.Sigmoid())

        self.network = nn.Sequential(*layers)

        # Xavier initialization for all linear layers
        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        return self.network(x)





# Main execution
def main():
    # Load and prepare data
    DATA_PATH = 'DeepLearning_Fall2025_hw4_prob1_data.npy'
    data = np.load(DATA_PATH)
    Y = data[:, 2]  # Shape: (60000,)
    X = data[:, 0:2]  # Shape: (60000, 2)

    print(f"X shape: {X.shape}")
    print(f"Y shape: {Y.shape}")

    # Normalize the data
    data_mean = X.mean(axis=0)
    data_std = X.std(axis=0)
    X_normalized = (X - data_mean) / data_std

    # Split data (first 50000 for training, last 10000 for testing)
    train_data = X_normalized[:50000]
    train_labels = Y[:50000]
    test_data = X_normalized[50000:]
    test_labels = Y[50000:]

    # Convert to PyTorch tensors and reshape labels to (n, 1)
    train_data_tensor = torch.FloatTensor(train_data)
    train_labels_tensor = torch.FloatTensor(train_labels).reshape(-1, 1)
    test_data_tensor = torch.FloatTensor(test_data)
    test_labels_tensor = torch.FloatTensor(test_labels).reshape(-1, 1)

    # Create datasets and dataloaders
    train_dataset = TensorDataset(train_data_tensor, train_labels_tensor)
    test_dataset = TensorDataset(test_data_tensor, test_labels_tensor)

    # Hyperparameters
    batch_size = 128
    learning_rate = 0.1
    epochs = 500
    seeds = [42, 123, 456, 789, 999]

    # Define different architectures to test
    architectures = {
        "Increased Width": [128, 64],  # Wider but same depth
        "Increased Depth": [32, 16, 8, 4],  # Deeper but similar total capacity
        "Large Capacity": [256, 128, 64, 32],  # Both wider and deeper
    }

    results = {}

    for arch_name, hidden_dims in architectures.items():
        print(f"\n{'='*50}")
        print(f"Testing Architecture: {arch_name}")
        print(f"Hidden Dimensions: {hidden_dims}")
        print(f"Depth: {len(hidden_dims)} layers")
        print(f"Widths: {hidden_dims}")
        print(f"{'='*50}")

        train_accuracies = []
        test_accuracies = []

        for i, seed in enumerate(seeds):
            print(f"\n--- Training with seed {seed} ---")
            set_seed(seed)

            # Create model and dataloaders
            model = FlexibleMLP(input_dim=2, hidden_dims=hidden_dims, output_dim=1)
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

            # Train model
            train_losses, test_accs = train_model(model, train_loader, test_loader, epochs, learning_rate)

            # Final evaluation
            train_acc = evaluate_model(model, train_data_tensor, train_labels_tensor)
            test_acc = evaluate_model(model, test_data_tensor, test_labels_tensor)

            train_accuracies.append(train_acc)
            test_accuracies.append(test_acc)

            print(f"Final Train Accuracy: {train_acc:.4f}")
            print(f"Final Test Accuracy: {test_acc:.4f}")

            # Plot training curves and visualize predictions for the last seed
            if i == len(seeds) - 1:
                plot_results(train_losses, test_accs, f"{arch_name} (Seed {seed})")
                visualize_predictions(model, test_data_tensor, test_labels_tensor, f"{arch_name} (Seed {seed})")

        # Store results
        results[arch_name] = {
            'train_mean': np.mean(train_accuracies),
            'train_std': np.std(train_accuracies),
            'test_mean': np.mean(test_accuracies),
            'test_std': np.std(test_accuracies),
            'train_accuracies': train_accuracies,
            'test_accuracies': test_accuracies
        }

        # Print summary for this architecture
        print(f"\n=== {arch_name} Summary ===")
        print(f"Train Accuracies: {[f'{acc:.4f}' for acc in train_accuracies]}")
        print(f"Test Accuracies: {[f'{acc:.4f}' for acc in test_accuracies]}")
        print(f"Mean Train Accuracy: {results[arch_name]['train_mean']:.4f} ± {results[arch_name]['train_std']:.4f}")
        print(f"Mean Test Accuracy: {results[arch_name]['test_mean']:.4f} ± {results[arch_name]['test_std']:.4f}")

    # Final comparison
    print(f"\n{'='*60}")
    print("FINAL COMPARISON ACROSS ARCHITECTURES")
    print(f"{'='*60}")
    for arch_name in architectures.keys():
        res = results[arch_name]
        print(f"{arch_name:20} | Train: {res['train_mean']:.4f} ± {res['train_std']:.4f} | Test: {res['test_mean']:.4f} ± {res['test_std']:.4f}")

main()